# Parole vs significato — perché ricerca ibrida (e perché non basta una soglia)

Prima di fidarsi dei conteggi per tema, una domanda di **metodo**: come si trova
un documento che parla di un tema? Per **parole** (BM25/full-text) o per
**significato** (embedding)? E si può classificare "ne parla / non ne parla" con
una **soglia** di similarità?

Questi esperimenti rispondono — e spiegano perché il vector database cerca in modo
**ibrido** (parole + significato fusi con RRF) e perché le analisi confrontano
distribuzioni a parità di volume invece di usare soglie.

## Come gira

Tutto sull'indice LanceDB reale, via `ponte.py` (vedi notebook 01 per il setup).

In [ ]:
import collections
import re
from ponte import vdb, tabella
from lancedb.rerankers import RRFReranker

emb = vdb.Embedder()
tab = tabella()

def doc_distinti(rows, k=15):
    out, visti = [], set()
    for r in rows:
        u = r.get("url") or r["titolo"]
        if u in visti:
            continue
        visti.add(u); out.append(u)
        if len(out) >= k:
            break
    return out

## I tre modi di cercare

Stessa query, tre ricerche: solo **vettori** (significato), solo **FTS** (parole,
il BM25 nativo di LanceDB) e **ibrido** (i due fusi con RRF). Poi guardiamo quanti
documenti trovano *in comune* e quanti *solo uno*.

In [ ]:
q = "la salvaguardia del pianeta e dell ecologia"
N = 40

vec = doc_distinti(tab.search(emb.query(q).astype("float32"))
                   .where("escludibile=false").limit(N).to_list())
fts = doc_distinti(tab.search(q, query_type="fts")
                   .where("escludibile=false").limit(N).to_list())
hyb = doc_distinti(tab.search(query_type="hybrid").vector(emb.query(q)).text(q)
                   .where("escludibile=false", prefilter=True)
                   .limit(N).rerank(RRFReranker()).to_list())

sv, sf = set(vec), set(fts)
print(f"top-15 documenti distinti per metodo, query: «{q}»")
print(f"  entrambi (vettori e FTS): {len(sv & sf)}")
print(f"  solo vettori            : {len(sv - sf)}")
print(f"  solo FTS (parole)       : {len(sf - sv)}")
print(f"  l'ibrido pesca: {sum(u in sv for u in hyb)} dal lato vettori, "
      f"{sum(u in sf for u in hyb)} dal lato parole")

### Risultato — complementarità

| | documenti (top-15) |
|---|--:|
| solo vettori | 13 |
| solo FTS (parole) | 13 |
| entrambi | **2** |

Su 15 documenti per metodo, solo **2 in comune**: i due modi pescano testi in gran
parte *diversi*. I vettori trovano "casa comune", "madre terra", "creato" senza la
parola "ecologia"; le parole prendono il match esatto. L'**ibrido** tiene il
meglio dei due (qui 7 documenti dal lato vettori, 9 dal lato parole). Ecco perché
la ricerca è ibrida. *(numeri sull'indice reale; variano un po' a ogni query)*

## Perché non una soglia

Tentazione naturale: marcare "parla di X" i documenti con similarità sopra una
soglia. **Non funziona.** Le similarità coseno di e5 non sono calibrate: stanno
tutte schiacciate in alto (rilevanti ~0,84, sfondo ~0,78) e la soglia che separa
un tema non separa l'altro. Il guadagno degli embedding non è classificare il
singolo documento, ma **recuperare** i passaggi giusti e dare **precisione**.

Per questo le analisi (notebook 01) non usano soglie: prendono i **top-N a parità
di volume** dei positivi regex, e confrontano le *distribuzioni* per Papa. Stesso
aggregato del conteggio a parole, ma documenti in buona parte diversi: a riprova,
sul tema *ambiente* regex e semantico si sovrappongono solo per ~37% dei documenti,
eppure danno lo stesso quadro per Papa. Solido.

## Footprint: ne parla solo un Papa?

Per scaldare lo strumento, due temi "di prova": il **calcio** (leggero) e
l'**ambiente** (la domanda seria: *"è roba solo di Francesco, e un po' comunista?"*).
Contiamo, per Papa, quanti documenti nominano il tema (parola chiave sull'indice,
saluti esclusi) e che quota sono del suo corpus.

In [ ]:
t = tab.search().where("escludibile=false").select(["url", "papa", "testo"]).limit(10**9).to_arrow()
urls = t.column("url").to_pylist(); papi = t.column("papa").to_pylist(); testi = t.column("testo").to_pylist()
dc, dp = collections.defaultdict(list), {}
for i, u in enumerate(urls):
    dc[u].append(i); dp[u] = papi[i]
tot = collections.Counter(dp.values())
PAPI = ["Papa Giovanni Paolo II", "Papa Benedetto XVI", "Papa Francesco", "Papa Leone XIV"]

def footprint(rx_str):
    rx = re.compile(rx_str, re.I)
    hit = {u for u in dc if any(rx.search(testi[i]) for i in dc[u])}
    c = collections.Counter(dp[u] for u in hit)
    for p in PAPI:
        print(f"  {p:26} {c[p]:4}  ({100*c[p]/tot[p]:4.1f}% del suo corpus)")

print("CALCIO"); footprint(r"calcio|football|soccer")
print("AMBIENTE"); footprint(r"ambient|ecolog|casa comune")

### Risultato — footprint per Papa

**Calcio** (documenti, % del corpus del Papa):

| Papa | doc | % |
|---|--:|--:|
| Giovanni Paolo II | 51 | 0,3% |
| Benedetto XVI | 18 | 0,6% |
| Francesco | 83 | 1,4% |
| Leone XIV | 2 | 0,3% |

Ne parlano un po' tutti: GP2 riceveva mezza Serie A, Francesco parla di Mondiali e
arbitri. Non è tema di nessuno in esclusiva.

**Ambiente** (regex, dal notebook 01): GP2 18% · BXVI 16% · FRA 19% · LEO 18% —
quasi identico per tutti. Non l'ha inventato Francesco: di suo c'è solo il *modo di
dirlo*, la formula "casa comune" della *Laudato si'*. Tema continuo, non
comunismo.

---
*Solo viste **aggregate** (conteggi e footprint), nessun testo ripubblicato (© LEV,
fonte `url` nei metadati). La lezione di metodo: i temi sfumati si misurano per
**significato**, la ricerca giusta è **ibrida**, e per confrontare i Papi si
guardano **distribuzioni** a parità di volume, non soglie.*